[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C3/notebooks/raman.ipynb)

# Odečtení pozadí Ramanova spektra — samostatná úloha

Ramanovo spektrum $\mathbf y$ má 300 hodnot na vlnočtech 400 až 1800 cm⁻¹. Pod úzkými
píky leží široké **pozadí z fluorescence**, které je potřeba odečíst, než se změří
výšky čtyř píků (u 1003, 1250, 1450 a 1660 cm⁻¹). O pozadí $\mathbf z$ víme tři věci:

1. pozadí **nikde neleží nad naměřeným spektrem** $\mathbf y$;
2. mezi všemi takovými křivkami chceme tu, která je spektru **co nejblíž**;
3. pozadí je **hladké** — mění se pomalu, nemá hrbolky jako píky.

Požadavky 2 a 3 jdou proti sobě, spojíme je **váhou** $\alpha = 10^4$ (jako alfa
u ridge v přednášce). $\alpha$ volíme my, **není to Lagrangeův multiplikátor**.

## Postup

1. **Tip:** kde se pozadí dotkne spektra? (Hřiště v reportu.)
2. Přepište zadání do tvaru minimize / subject to, doplňte model a vyřešte ho *(díra 1)*.
3. Výšky píků a body dotyku z multiplikátorů *(díra 2)*.
4. Zkuste jinou váhu α a odpovězte slovy.

---

*Čísla, se kterými se notebook porovnává, počítá skript [`kod/raman.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C3/kod/raman.py).*

<details><summary><b>Nápověda k přepisu</b> (otevřít, když se zaseknete)</summary>

| věta ze zadání | co z ní bude v modelu |
|---|---|
| „leží pod signálem“ | **omezení**, ne člen v cíli: $z_i \le y_i$ pro každý bod, v CVXPY jeden řádek pro celý vektor |
| „co nejblíž“ | plocha mezi křivkami $\sum_i (y_i - z_i)$; díky omezení je každý rozdíl nezáporný, absolutní hodnota netřeba |
| „hladké“ | malé druhé diference $z_{i+1} - 2z_i + z_{i-1}$, tedy trest $\alpha\,\lVert D_2\mathbf z\rVert_2^2$; matice $D_2$ je `np.diff(np.eye(n), 2, axis=0)` |

</details>

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
# Simulace spektra (stejná data jako v reportu). Tuhle buňku nemusíte číst,
# stačí ji spustit: dává spektrum y, vlnočty v a indexy středů píků INDEXY_PIKU.
import numpy as np
import matplotlib.pyplot as plt

n = 300
v = np.linspace(400, 1800, n)                       # Ramanův posun [cm^-1]
rng = np.random.default_rng(1)
pozadi_pravda = 3 * np.exp(-(v - 400) / 900) + 0.8  # v laboratoři ho neznáme
PIKY = [(1003, 4.0, 8), (1250, 1.5, 25), (1450, 2.5, 15), (1660, 2.0, 20)]  # střed, výška, sigma
piky = sum(h * np.exp(-(v - c) ** 2 / (2 * s ** 2)) for c, h, s in PIKY)
y = pozadi_pravda + piky + rng.normal(0, 0.05, n)  # šum měření sigma = 0,05
INDEXY_PIKU = [int(np.argmin(np.abs(v - c))) for c, _, _ in PIKY]

## Krok 1 — model v CVXPY *(díra 1)*

In [ ]:
alfa = 1e4   # váha hladkosti; zkuste 1e2 a 1e6

In [ ]:
# TODO 1: doplňte matici druhých diferencí, omezení a účelovou funkci.
D2 = ...      # nápověda: np.diff(np.eye(n), 2, axis=0)
pozadi = cp.Variable(n)
pod = ...     # pozadí nikde nad spektrem
omezeni = [pod]
uloha = cp.Problem(cp.Minimize(...), omezeni)   # „co nejblíž“ + alfa * „hladké“

In [ ]:
uloha.solve(solver=cp.CLARABEL)
print("stav:", uloha.status)
assert uloha.status == "optimal", "unbounded? zkontrolujte směr omezení a znaménko v cíli"
rmse = np.sqrt(np.mean((pozadi.value - pozadi_pravda) ** 2))   # typická odchylka od skutečného pozadí
print(f"alfa = {alfa:g}: RMSE pozadí vůči skutečnému {rmse:.3f}")
if alfa == 1e4:                      # kontrola jen pro zadanou váhu, jiné jen vypíšou
    assert abs(rmse - 0.083) < 0.002

## Krok 2 — výšky píků a komplementarita *(díra 2)*

Výška píku = hodnota $\mathbf y - \mathbf z$ v bodě nejbližším ke středu píku. Je v ní
i šum měření (σ = 0,05), takže rozdíly kolem 0,03–0,06 proti pravdě nejsou chyba metody.
Multiplikátor $\mu_i$ omezení $z_i \le y_i$ je podle komplementarity nenulový jen tam,
kde se pozadí spektra dotýká. Solver vrací i drobné nenuly, proto práh 0,1.

In [ ]:
# TODO 2: korigované spektrum, výšky píků a body dotyku.
korigovane = ...                  # spektrum minus pozadí
vysky = ...                       # korigovane v bodech INDEXY_PIKU
mu = pod.dual_value
dotyky = ...                      # indexy bodů s mu > 0.1, nápověda: np.flatnonzero(...)

In [ ]:
print("výšky píků:", np.round(vysky, 2), "(pravda 4,0 / 1,5 / 2,5 / 2,0)")
print(f"dotyků: {len(dotyky)}, na vlnočtech {np.round(v[dotyky]).astype(int)}")
print(f"komplementarita max |mu_i (y_i - z_i)| = {np.max(np.abs(mu * korigovane)):.1e}")
if alfa == 1e4:
    assert np.allclose(vysky, [4.03, 1.26, 2.44, 1.82], atol=0.01)
    assert len(dotyky) == 25

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.plot(v, y, label="spektrum y")
ax1.plot(v, pozadi.value, label="pozadí z")
ax1.plot(v[dotyky], y[dotyky], "o", label="dotyky (mu > 0,1)")
ax1.set_title(f"alfa = {alfa:g}")
ax1.legend()
ax2.plot(v, korigovane, label="y - z")
ax2.plot(v[INDEXY_PIKU], vysky, "o", label="výšky píků")
ax2.set_xlabel("Ramanův posun [cm$^{-1}$]")
ax2.legend()
plt.show()

## Krok 3 — odpovězte slovy

1. Změňte v buňce `alfa = …` (nad modelem) váhu na `1e2` a pak na `1e6` a spusťte
   ji a buňky pod ní až po graf. Co dělá pozadí pod píky?
2. Kde se pozadí dotýká spektra a proč zrovna tam?

**TODO:** 1. … 2. …